# NYC Taxi Vulkan GPU Machine Learning (`vkpandas` + `vklearn`)

End-to-end GPU machine learning on real NYC Taxi trips running on **`vknumpy` + `vkpandas` + `vklearn`** (`ggml-vulkan`): fare regression (`Ridge`), tip classification (`LogisticRegression` Newton-IRLS), **tip amount (`$`) regression (`Ridge`)**, pickup hotspot clustering (`KMeans`), and live interactive trip prediction.

In [1]:
import math
import sys
import time
from pathlib import Path

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
for _p in (_root, _root.parent / "vkpandas", _root.parent / "delightful-hertz"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import numpy as np
import vknumpy as vnp
import vkpandas as vpd
import vklearn as vkl

print(f"Vulkan device: {vnp.device_info()['name']} ({vnp.device_info()['backend']})")

PARQUET = Path.home() / ".cache/hvsampledata/nyc_taxi/v2/nyc_taxi_wide.parq"
t0 = time.perf_counter()
df = vpd.read_parquet(
    PARQUET,
    columns=["pickup_x", "pickup_y", "dropoff_x", "dropoff_y", "trip_distance", "fare_amount", "tip_amount", "pickup_hour"],
)
valid = (
    (df["fare_amount"] > 2.5) & (df["fare_amount"] < 100.0) &
    (df["trip_distance"] > 0.1) & (df["trip_distance"] < 50.0) &
    (df["pickup_x"] >= -8242000.0) & (df["pickup_x"] <= -8226000.0) &
    (df["pickup_y"] >= 4968000.0) & (df["pickup_y"] <= 4985000.0)
)
clean = df[valid]
print(f"Loaded & GPU-filtered {len(clean):,} valid trips in {time.perf_counter() - t0:.2f}s")

# Use 1,000,000 trips for fast interactive training
sub = clean[:1_000_000]
FEATURES = ["trip_distance", "pickup_x", "pickup_y", "dropoff_x", "dropoff_y", "pickup_hour"]
X = sub[FEATURES]
y_fare = sub["fare_amount"]
y_tip_amt = sub["tip_amount"]
y_tipped = (y_tip_amt > 0.0).astype(vnp.int32)

X_tr, X_te, yf_tr, yf_te, yta_tr, yta_te, yt_tr, yt_te = vkl.train_test_split(
    X, y_fare, y_tip_amt, y_tipped, test_size=0.2, random_state=42
)
print(f"Train: {len(X_tr):,} rows | Test: {len(X_te):,} rows")

ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


Vulkan device: AMD Custom GPU 0405 (RADV VANGOGH) (ggml-vulkan)


Loaded & GPU-filtered 11,400,404 valid trips in 2.44s


Train: 800,000 rows | Test: 200,000 rows


## 1. GPU Ridge Regression — Fare Prediction

In [2]:
t0 = time.perf_counter()
fare_pipe = vkl.Pipeline([
    ("scaler", vkl.StandardScaler()),
    ("ridge", vkl.Ridge(alpha=1.0)),
]).fit(X_tr, yf_tr)
pred_fare = fare_pipe.predict(X_te)
dt = time.perf_counter() - t0

print(f"GPU Ridge fit + predict in {dt:.3f}s")
print(f"  R^2  = {vkl.r2_score(yf_te, pred_fare):.4f}")
print(f"  RMSE = ${vkl.root_mean_squared_error(yf_te, pred_fare):.2f}")
print(f"  MAE  = ${vkl.mean_absolute_error(yf_te, pred_fare):.2f}")

GPU Ridge fit + predict in 0.154s
  R^2  = 0.8780
  RMSE = $2.02
  MAE  = $1.25


## 2. GPU Logistic Regression (Newton-IRLS) — Tip Classifier

In [3]:
t0 = time.perf_counter()
tip_pipe = vkl.Pipeline([
    ("scaler", vkl.StandardScaler()),
    ("logreg", vkl.LogisticRegression(C=1.0, max_iter=15)),
]).fit(X_tr, yt_tr)
pred_tip = tip_pipe.predict(X_te)
dt = time.perf_counter() - t0
cm = vkl.confusion_matrix(yt_te, pred_tip, num_classes=2).to_numpy()

print(f"GPU LogisticRegression ({tip_pipe.named_steps['logreg'].n_iter_} IRLS iters) in {dt:.3f}s")
print(f"  Accuracy = {vkl.accuracy_score(yt_te, pred_tip) * 100:.2f}%")
print(f"  Confusion Matrix = {cm.tolist()}")

GPU LogisticRegression (4 IRLS iters) in 0.242s
  Accuracy = 60.54%
  Confusion Matrix = [[4722, 75203], [3716, 116359]]


[0 AMD Custom GPU 0405 (RADV VANGOGH)]  queueC=1[4]  queueT=0[1]  prefer-slm=0  rebar=1  r-score=11
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-p/s/u/a=1/1/1/1  int8-p/s/u/a=1/1/1/1  bf16-p/s=1/0
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  subgroup=64(32~64)  ops=1/1/1/1/1/1/1/1/1/1
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-cm=0  int8-cm=0  bf16-cm=0  fp8-cm=0  fp6-cm=0  fp4-cm=0


## 2b. GPU Ridge Regression — Tip Amount ($) Prediction

In NYC TLC taxi data, cash tips are recorded as `$0.00`, while credit-card tips average ~15–22% of the fare. Here we train **both**:
- `tip_amt_pipe`: overall expected tip (`$`) across all trips
- `tip_card_pipe`: conditional tip (`$`) given the rider tips (`tip_amount > 0`)

In [4]:
t0 = time.perf_counter()
tip_amt_pipe = vkl.Pipeline([
    ("scaler", vkl.StandardScaler()),
    ("ridge", vkl.Ridge(alpha=1.0)),
]).fit(X_tr, yta_tr)
pred_tip_all = tip_amt_pipe.predict(X_te)

tipped_mask_tr = yta_tr > 0.0
tipped_mask_te = yta_te > 0.0
tip_card_pipe = vkl.Pipeline([
    ("scaler", vkl.StandardScaler()),
    ("ridge", vkl.Ridge(alpha=1.0)),
]).fit(X_tr[tipped_mask_tr], yta_tr[tipped_mask_tr])
pred_tip_card = tip_card_pipe.predict(X_te[tipped_mask_te])
dt = time.perf_counter() - t0

print(f"GPU Tip Amount Ridge (both models) in {dt:.3f}s")
print(f"  All trips:    R^2 = {vkl.r2_score(yta_te, pred_tip_all):.4f} | RMSE = ${vkl.root_mean_squared_error(yta_te, pred_tip_all):.2f} | MAE = ${vkl.mean_absolute_error(yta_te, pred_tip_all):.2f}")
print(f"  Tipped trips: R^2 = {vkl.r2_score(yta_te[tipped_mask_te], pred_tip_card):.4f} | RMSE = ${vkl.root_mean_squared_error(yta_te[tipped_mask_te], pred_tip_card):.2f} | MAE = ${vkl.mean_absolute_error(yta_te[tipped_mask_te], pred_tip_card):.2f}")

GPU Tip Amount Ridge (both models) in 0.294s
  All trips:    R^2 = 0.2020 | RMSE = $1.46 | MAE = $1.04
  Tipped trips: R^2 = 0.4342 | RMSE = $1.20 | MAE = $0.56


## 3. GPU K-Means — 8 Manhattan Pickup Hotspot Clusters

In [5]:
t0 = time.perf_counter()
coords = sub[["pickup_x", "pickup_y"]]
km = vkl.KMeans(n_clusters=8, max_iter=20, random_state=42).fit(coords)
dt = time.perf_counter() - t0
counts = vnp.bincount(km.labels_, minlength=8).to_numpy()
print(f"GPU KMeans (K=8 on 1,000,000 pickups, {km.n_iter_} iters) in {dt:.2f}s")
print(f"  Cluster sizes: {counts.tolist()}")

GPU KMeans (K=8 on 1,000,000 pickups, 20 iters) in 8.46s
  Cluster sizes: [62919, 158627, 42673, 142537, 139451, 68555, 169480, 215758]


## 4. Predict Fare, Tip Amount ($), Tip Probability & Pickup Cluster from Custom Input Variables

In [6]:
def lonlat_to_mercator(lon: float, lat: float) -> tuple[float, float]:
    r = 6378137.0
    x = math.radians(lon) * r
    y = math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0)) * r
    return x, y

# --- Edit these input variables to predict any NYC taxi trip! ---
trip_distance = 4.2        # miles
pickup_lon, pickup_lat   = -73.9855, 40.7580   # Times Square
dropoff_lon, dropoff_lat = -74.0113, 40.7069   # Wall Street
pickup_hour = 18           # 0..23 (6 PM)

px, py = lonlat_to_mercator(pickup_lon, pickup_lat)
dx, dy = lonlat_to_mercator(dropoff_lon, dropoff_lat)

X_new = vpd.DataFrame({
    "trip_distance": [trip_distance],
    "pickup_x": [px],
    "pickup_y": [py],
    "dropoff_x": [dx],
    "dropoff_y": [dy],
    "pickup_hour": [pickup_hour],
})

pred_dollars = float(fare_pipe.predict(X_new).to_numpy()[0])
pred_tip_card_d = max(0.0, float(tip_card_pipe.predict(X_new).to_numpy()[0]))
pred_tip_all_d = max(0.0, float(tip_amt_pipe.predict(X_new).to_numpy()[0]))
tip_prob = float(tip_pipe.named_steps["logreg"].predict_proba(tip_pipe.named_steps["scaler"].transform(X_new)).to_numpy()[0, 1])
cluster_id = int(km.predict(X_new[["pickup_x", "pickup_y"]]).to_numpy()[0])

print(f"Predicted Fare:             ${pred_dollars:.2f}")
print(f"Predicted Tip (Card):       ${pred_tip_card_d:.2f}  -> Total: ${pred_dollars + pred_tip_card_d:.2f}")
print(f"Dataset Avg Tip (All):      ${pred_tip_all_d:.2f}")
print(f"Card-Tip Probability (> $0): {tip_prob * 100:.1f}%")
print(f"Pickup Hotspot Cluster:     #{cluster_id} (cluster size = {int(counts[cluster_id]):,} trips)")

Predicted Fare:             $16.90
Predicted Tip (Card):       $3.28  -> Total: $20.18
Dataset Avg Tip (All):      $2.29
Card-Tip Probability (> $0): 71.6%
Pickup Hotspot Cluster:     #3 (cluster size = 142,537 trips)


In [7]:
import ipywidgets as widgets
from IPython.display import display

LANDMARKS = {
    "Times Square": (-73.9855, 40.7580),
    "Wall Street": (-74.0113, 40.7069),
    "Central Park": (-73.9654, 40.7829),
    "SoHo": (-74.0005, 40.7233),
    "Upper East Side": (-73.9565, 40.7736),
    "Williamsburg (Brooklyn)": (-73.9571, 40.7081),
    "Penn Station": (-73.9935, 40.7506),
}

dd_pick = widgets.Dropdown(options=list(LANDMARKS.keys()), value="Times Square", description="Pickup:")
dd_drop = widgets.Dropdown(options=list(LANDMARKS.keys()), value="Wall Street", description="Dropoff:")
sl_dist = widgets.FloatSlider(value=4.2, min=0.2, max=25.0, step=0.1, description="Miles:")
sl_hour = widgets.IntSlider(value=18, min=0, max=23, step=1, description="Hour:")
out_lbl = widgets.HTML()

def _predict_live(_=None) -> None:
    px, py = lonlat_to_mercator(*LANDMARKS[dd_pick.value])
    dx, dy = lonlat_to_mercator(*LANDMARKS[dd_drop.value])
    row = vpd.DataFrame({
        "trip_distance": [sl_dist.value],
        "pickup_x": [px], "pickup_y": [py],
        "dropoff_x": [dx], "dropoff_y": [dy],
        "pickup_hour": [sl_hour.value],
    })
    fare = float(fare_pipe.predict(row).to_numpy()[0])
    tip_card = max(0.0, float(tip_card_pipe.predict(row).to_numpy()[0]))
    tip_all = max(0.0, float(tip_amt_pipe.predict(row).to_numpy()[0]))
    p_tip = float(tip_pipe.named_steps["logreg"].predict_proba(tip_pipe.named_steps["scaler"].transform(row)).to_numpy()[0, 1])
    cid = int(km.predict(row[["pickup_x", "pickup_y"]]).to_numpy()[0])
    out_lbl.value = (
        f"<div style='padding:10px;background:#181b24;color:#e5e9f0;border-radius:6px;font-family:monospace'>"
        f"<b>Fare:</b> <span style='color:#ff9128;font-size:16px'>${fare:.2f}</span> &nbsp;|&nbsp; "
        f"<b>Card Tip:</b> <span style='color:#7ee787;font-size:16px'>${tip_card:.2f}</span> &nbsp;|&nbsp; "
        f"<b>Avg Tip (All):</b> <span style='color:#ffd580;font-size:16px'>${tip_all:.2f}</span> &nbsp;|&nbsp; "
        f"<b>Card-Tip Prob:</b> <span style='color:#50c3ff'>{p_tip*100:.1f}%</span> &nbsp;|&nbsp; "
        f"<b>Cluster:</b> #{cid}</div>"
    )

for w in (dd_pick, dd_drop, sl_dist, sl_hour):
    w.observe(_predict_live, names="value")
_predict_live()
display(widgets.VBox([widgets.HBox([dd_pick, dd_drop]), widgets.HBox([sl_dist, sl_hour]), out_lbl]))